# NumPydantic v2 Demo

A quick walkthrough of this library:

1. Validate **dtype** and **shape** on input
2. Serialize to a human-readable JSON list (default)
3. Round-trip exactly, preserving dtype and shape
4. Serialize to a compact base64-encoded payload

In [2]:
import numpy as np
from pydantic import BaseModel, ValidationError

from numpydantic import NDArray

### 1. Define a typed model

`NDArray(dtype=..., shape=...)` returns a Pydantic-compatible type. `None` entries in the shape leave that axis unconstrained.

In [9]:
class Matrix(BaseModel):
    data: NDArray(dtype="float64", shape=(None, 3))   # any number of rows, 3 columns


m = Matrix(data=[[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
print(m.data)
print("shape:", m.data.shape, " dtype:", m.data.dtype)

[[1. 2. 3.]
 [4. 5. 6.]]
shape: (2, 3)  dtype: float64


### 2. Default `list` mode: human-readable

`model_dump_json()` returns a bare nested list. Easy to inspect by hand, but dtype information is lost on a round trip (the restored array picks numpy's default int/float).

In [10]:
print(m.model_dump_json())

{"data":[[1.0,2.0,3.0],[4.0,5.0,6.0]]}


### 3. `list` mode with `round_trip=True` : preserves dtype + shape

Pydantic's built-in `round_trip=True` flag switches `list` mode to a metadata envelope (`{array, dtype, shape}`) so the original dtype and shape can be recovered on validation.

In [11]:
payload = m.model_dump_json(round_trip=True)
print("envelope:", payload)

restored = Matrix.model_validate_json(payload)
print("restored:", restored.data)
print("dtype preserved:", restored.data.dtype == m.data.dtype)

envelope: {"data":{"array":[[1.0,2.0,3.0],[4.0,5.0,6.0]],"dtype":"float64","shape":[2,3]}}
restored: [[1. 2. 3.]
 [4. 5. 6.]]
dtype preserved: True


### 4. `base64` mode : compact + bit-exact

Base64 always returns the metadata envelope and stores the raw byte buffer for the payload. It's the right choice for performance-sensitive situations.
> see the **Performance** section in the README for size and speed comparisons against list mode.

It also preserves floats bit-exactly, so `NaN` and `±inf` survive a JSON round trip.

In [6]:
class ExactModel(BaseModel):
    values: NDArray(dtype="float64", encoding="base64")


# NaN and infinity round-trip exactly through JSON.
e = ExactModel(values=np.array([np.nan, np.inf, -np.inf, 1.5]))
print("dump:", e.model_dump_json())

restored = ExactModel.model_validate_json(e.model_dump_json())
print("restored:", restored.values)
print("equal (NaN-aware):", np.array_equal(restored.values, e.values, equal_nan=True))

dump: {"values":{"array":"AAAAAAAA+H8AAAAAAADwfwAAAAAAAPD/AAAAAAAA+D8=","dtype":"float64","shape":[4],"encoding":"base64"}}
restored: [ nan  inf -inf  1.5]
equal (NaN-aware): True
